# Kaggle Competition: Real vs. Non-Disaster Tweets
### DSC680-T301 Applied Data Science
### Nicholas Stirling
#### Objective
The goal of this notebook is to build a machine learning pipeline capable of classifying tweets as either:
- **1** → Real disaster
- **0** → Not a real disaster   
This is a classic **binary text classification problem** within Natural Language Processing (NLP).

We will follow a structured pipeline:
 1. Data Loading & Inspection
 2. Text Preprocessing
 3. Feature Engineering (TF-IDF)
 4. Model Training (Logistic Regression baseline)
 5. Evaluation
 6. Prediction & Submission

In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.head())
print(train.info())

   id keyword location                                               text  \
0   1     NaN      NaN  Our Deeds are the Reason of this #earthquake M...   
1   4     NaN      NaN             Forest fire near La Ronge Sask. Canada   
2   5     NaN      NaN  All residents asked to 'shelter in place' are ...   
3   6     NaN      NaN  13,000 people receive #wildfires evacuation or...   
4   7     NaN      NaN  Just got sent this photo from Ruby #Alaska as ...   

   target  
0       1  
1       1  
2       1  
3       1  
4       1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        7613 non-null   int64 
 1   keyword   7552 non-null   object
 2   location  5080 non-null   object
 3   text      7613 non-null   object
 4   target    7613 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 297.5+ KB
None


The raw text data from tweets likely contains noise in the form of URLs, punctuation, stopwords, and mixed cases. Cleaning will improve the models signal, which will be accomplished with lowercasing, removing urls, puctuation, and whitespace.

In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.strip()
    return text

train['clean_text'] = train['text'].apply(clean_text)
test['clean_text'] = test['text'].apply(clean_text)

After cleaning the text is normalized and ready for vectorization. From here we can convert the text into numberical vectors using TF-IDF. This will capture the importance of the words, reduce the weight of common words, and will work well with baseline NLP models. We will use both unigrams and bigrams and a set max feature limit to reduce dimensionality.

In [4]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=10000,
    stop_words='english'
)

X = vectorizer.fit_transform(train['clean_text'])
y = train['target']

X_test = vectorizer.transform(test['clean_text'])

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


The split of the data means each feature will now correspond to a feature or feature pair. The will allow linear models to perform well with the text data. We split the training data to evaluate model performance. This will prevent overfitting and allow us to generalize the model performance. We use Logistic Regression as a baseline model as it is fast to train, is easily interpretable, and will present a strong baseline for text classification. We should see a stable baseline performance with the model. To evaluate this we will use Accuracy, Precision, Recall, and F1-score for the classification balance.

In [7]:
y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.7997373604727511
              precision    recall  f1-score   support

           0       0.79      0.90      0.84       874
           1       0.83      0.67      0.74       649

    accuracy                           0.80      1523
   macro avg       0.81      0.78      0.79      1523
weighted avg       0.80      0.80      0.80      1523



Evaluation shows the model has strong overall performance. The model is very effective at identifying non-disaster tweets, with a high recall of 0.90. This means it rarely misclassifies normal tweets as disasters. However, it also misses a meaningful portion of the actual disaster tweets, recall of 0.67. False negatives, missed disasters, can be costly for this type of analysis as opposed to false positives. The model is currently biased toward the majority class, non-disaster tweets, which is common in imbalanced NLP classification tasks.  
While an 80% accuracy is a strong baseline, the recall gap for disaster tweets highlights the need for refinement. This can be accomplished with class weighting to address the misclassification of disaster tweets, threshold tuning from the default of 0.5, and feature expansion like adding domain specific keywords.

In [9]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

probs = model.predict_proba(X_val)[:, 1]

In [10]:
thresholds = np.arange(0.3, 0.7, 0.05)
results = []

from sklearn.metrics import f1_score, recall_score, precision_score

for t in thresholds:
    preds = (probs >= t).astype(int)
    results.append({
        'threshold': t,
        'precision': precision_score(y_val, preds),
        'recall': recall_score(y_val, preds),
        'f1': f1_score(y_val, preds)
    })

results_df = pd.DataFrame(results)
print(results_df)

   threshold  precision    recall        f1
0       0.30   0.532743  0.927581  0.676785
1       0.35   0.589069  0.896764  0.711057
2       0.40   0.633218  0.845917  0.724274
3       0.45   0.684421  0.791988  0.734286
4       0.50   0.750000  0.739599  0.744763
5       0.55   0.816176  0.684129  0.744342
6       0.60   0.881432  0.607088  0.718978
7       0.65   0.915966  0.503852  0.650099


In [15]:
best_threshold = 0.35

y_pred = (probs >= best_threshold).astype(int)

print("Using threshold:", best_threshold)
print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Using threshold: 0.35
Accuracy: 0.6894287590282338
              precision    recall  f1-score   support

           0       0.87      0.54      0.66       874
           1       0.59      0.90      0.71       649

    accuracy                           0.69      1523
   macro avg       0.73      0.72      0.69      1523
weighted avg       0.75      0.69      0.68      1523



In [17]:
y_pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

Accuracy: 0.7839789888378201
              precision    recall  f1-score   support

           0       0.81      0.82      0.81       874
           1       0.75      0.74      0.74       649

    accuracy                           0.78      1523
   macro avg       0.78      0.78      0.78      1523
weighted avg       0.78      0.78      0.78      1523



Accuracy decreased overall but revall was aligned with the objective of prioritizing reducing false negatives. With this we achieved a 7% gain in recall for disaster detection, which better aligns with real-world risk tolerance.

In [19]:
test_preds = model.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'target': test_preds
})

submission.to_csv('submission.csv', index=False)

print("Submission file created.")

Submission file created.
